# Time Drift EDA (Fine-grained Segments + Holdout)

依據 `strategy/time_drift.md` 實作：

- Drift Track：細分時間段（預設 `N_SEGMENTS=12`）
- Model Track：保留最後 15% unique steps 為穩定 holdout
- 產出：
  - `drift_summary_fine_segments.csv`
  - `drift_plots/*.png`
  - `drift_findings.md`
  - `holdout_definition.md`


In [ ]:
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')


In [ ]:
DATA_PATH = Path('../Transactions Data.csv')
OUT_DIR = Path('../')
PLOTS_DIR = OUT_DIR / 'drift_plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

N_SEGMENTS = 12
CHUNK_SIZE = 500_000
HOLDOUT_RATIO = 0.15

AMOUNT_BINS = [-np.inf, 100, 1_000, 10_000, 100_000, np.inf]
AMOUNT_LABELS = ['<=100', '100-1k', '1k-10k', '10k-100k', '100k+']

NUM_COLS = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
SAMPLE_PER_SEG = 6000
RANDOM_STATE = 42


In [ ]:
step_series = pd.read_csv(DATA_PATH, usecols=['step'])['step']
unique_steps = np.sort(step_series.unique())

segments = np.array_split(unique_steps, N_SEGMENTS)
step_to_seg = {}
seg_bounds = []
for seg_id, arr in enumerate(segments):
    if len(arr) == 0:
        continue
    seg_min = int(arr.min())
    seg_max = int(arr.max())
    seg_bounds.append((seg_id, seg_min, seg_max, len(arr)))
    for s in arr:
        step_to_seg[int(s)] = seg_id

seg_df = pd.DataFrame(seg_bounds, columns=['segment_id', 'step_min', 'step_max', 'n_unique_steps'])

holdout_start_idx = int(len(unique_steps) * (1 - HOLDOUT_RATIO))
holdout_start_step = int(unique_steps[holdout_start_idx])

print('Unique steps:', len(unique_steps))
print('N segments:', len(seg_df))
print('Holdout start step:', holdout_start_step)
seg_df.head()


In [ ]:
seg_count = defaultdict(int)
seg_fraud = defaultdict(int)
seg_flagged = defaultdict(int)

seg_type_count = defaultdict(lambda: defaultdict(int))
seg_type_fraud = defaultdict(lambda: defaultdict(int))

seg_bucket_count = defaultdict(lambda: defaultdict(int))
seg_bucket_fraud = defaultdict(lambda: defaultdict(int))

seg_flag_fraud_cross = defaultdict(lambda: defaultdict(int))

rng = np.random.default_rng(RANDOM_STATE)
seg_samples = {seg_id: {c: np.array([], dtype=float) for c in NUM_COLS + ['deltaOrig','deltaDest','amount_log1p']} 
               for seg_id in seg_df['segment_id'].tolist()}

def reservoir_merge(old_arr, new_arr, cap, rng):
    if len(old_arr) == 0:
        if len(new_arr) <= cap:
            return new_arr
        idx = rng.choice(len(new_arr), size=cap, replace=False)
        return new_arr[idx]
    merged = np.concatenate([old_arr, new_arr])
    if len(merged) <= cap:
        return merged
    idx = rng.choice(len(merged), size=cap, replace=False)
    return merged[idx]

use_cols = ['step','type','amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFraud','isFlaggedFraud']

for chunk in pd.read_csv(DATA_PATH, usecols=use_cols, chunksize=CHUNK_SIZE):
    chunk['deltaOrig'] = chunk['oldbalanceOrg'] - chunk['newbalanceOrig']
    chunk['deltaDest'] = chunk['newbalanceDest'] - chunk['oldbalanceDest']
    chunk['amount_log1p'] = np.log1p(chunk['amount'].clip(lower=0))

    chunk['segment_id'] = chunk['step'].map(step_to_seg)
    chunk = chunk.dropna(subset=['segment_id']).copy()
    chunk['segment_id'] = chunk['segment_id'].astype(int)

    chunk['amount_bucket'] = pd.cut(chunk['amount'], bins=AMOUNT_BINS, labels=AMOUNT_LABELS, include_lowest=True)

    g = chunk.groupby('segment_id', observed=True)
    core = g.agg(rows=('isFraud','size'), fraud_sum=('isFraud','sum'), flagged_sum=('isFlaggedFraud','sum'))
    for seg_id, row in core.iterrows():
        seg_count[seg_id] += int(row['rows'])
        seg_fraud[seg_id] += int(row['fraud_sum'])
        seg_flagged[seg_id] += int(row['flagged_sum'])

    type_cnt = chunk.groupby(['segment_id','type'], observed=True).size().reset_index(name='n')
    type_frd = chunk.groupby(['segment_id','type'], observed=True)['isFraud'].sum().reset_index(name='f')
    for _, r in type_cnt.iterrows():
        seg_type_count[int(r['segment_id'])][str(r['type'])] += int(r['n'])
    for _, r in type_frd.iterrows():
        seg_type_fraud[int(r['segment_id'])][str(r['type'])] += int(r['f'])

    b_cnt = chunk.groupby(['segment_id','amount_bucket'], observed=True).size().reset_index(name='n')
    b_frd = chunk.groupby(['segment_id','amount_bucket'], observed=True)['isFraud'].sum().reset_index(name='f')
    for _, r in b_cnt.iterrows():
        seg_bucket_count[int(r['segment_id'])][str(r['amount_bucket'])] += int(r['n'])
    for _, r in b_frd.iterrows():
        seg_bucket_fraud[int(r['segment_id'])][str(r['amount_bucket'])] += int(r['f'])

    cross = chunk.groupby(['segment_id','isFlaggedFraud','isFraud'], observed=True).size().reset_index(name='n')
    for _, r in cross.iterrows():
        key = f"flag{int(r['isFlaggedFraud'])}_fraud{int(r['isFraud'])}"
        seg_flag_fraud_cross[int(r['segment_id'])][key] += int(r['n'])

    sample_cols = NUM_COLS + ['deltaOrig','deltaDest','amount_log1p']
    for seg_id, sdf in chunk.groupby('segment_id', observed=True):
        for c in sample_cols:
            arr = sdf[c].dropna().to_numpy(dtype=float)
            if len(arr) == 0:
                continue
            seg_samples[int(seg_id)][c] = reservoir_merge(seg_samples[int(seg_id)][c], arr, SAMPLE_PER_SEG, rng)

print('Chunk aggregation done.')


In [ ]:
rows = []
for seg_id in seg_df['segment_id']:
    n = seg_count.get(seg_id, 0)
    f = seg_fraud.get(seg_id, 0)
    flg = seg_flagged.get(seg_id, 0)

    row = {
        'segment_id': seg_id,
        'rows': n,
        'fraud_sum': f,
        'fraud_rate': (f / n) if n else np.nan,
        'flagged_sum': flg,
        'flagged_rate': (flg / n) if n else np.nan,
    }

    for c in NUM_COLS + ['deltaOrig','deltaDest','amount_log1p']:
        arr = seg_samples[seg_id][c]
        if len(arr) == 0:
            row[f'{c}_mean'] = np.nan
            row[f'{c}_median'] = np.nan
            row[f'{c}_p90'] = np.nan
            row[f'{c}_p99'] = np.nan
        else:
            row[f'{c}_mean'] = float(np.mean(arr))
            row[f'{c}_median'] = float(np.quantile(arr, 0.5))
            row[f'{c}_p90'] = float(np.quantile(arr, 0.9))
            row[f'{c}_p99'] = float(np.quantile(arr, 0.99))

    rows.append(row)

drift_summary = pd.DataFrame(rows).merge(seg_df, on='segment_id', how='left').sort_values('segment_id')
drift_summary['fraud_rate_roll3'] = drift_summary['fraud_rate'].rolling(3, min_periods=1).mean()
drift_summary.head()


In [ ]:
all_types = sorted({t for seg in seg_type_count.values() for t in seg.keys()})

type_share_rows = []
for seg_id in drift_summary['segment_id']:
    total = seg_count.get(seg_id, 0)
    for t in all_types:
        cnt = seg_type_count[seg_id].get(t, 0)
        frd = seg_type_fraud[seg_id].get(t, 0)
        type_share_rows.append({
            'segment_id': seg_id,
            'type': t,
            'count': cnt,
            'share': (cnt / total) if total else np.nan,
            'fraud_rate_in_type': (frd / cnt) if cnt else np.nan
        })

type_share_df = pd.DataFrame(type_share_rows)

bucket_rows = []
for seg_id in drift_summary['segment_id']:
    for b in AMOUNT_LABELS:
        cnt = seg_bucket_count[seg_id].get(str(b), 0)
        frd = seg_bucket_fraud[seg_id].get(str(b), 0)
        bucket_rows.append({
            'segment_id': seg_id,
            'amount_bucket': str(b),
            'count': cnt,
            'fraud_rate': (frd / cnt) if cnt else np.nan
        })
bucket_df = pd.DataFrame(bucket_rows)

cross_rows = []
for seg_id in drift_summary['segment_id']:
    d = seg_flag_fraud_cross[seg_id]
    n00 = d.get('flag0_fraud0', 0)
    n01 = d.get('flag0_fraud1', 0)
    n10 = d.get('flag1_fraud0', 0)
    n11 = d.get('flag1_fraud1', 0)
    cross_rows.append({
        'segment_id': seg_id,
        'n00': n00, 'n01': n01, 'n10': n10, 'n11': n11,
        'p_fraud_given_flag0': n01 / max((n00+n01), 1),
        'p_fraud_given_flag1': n11 / max((n10+n11), 1)
    })
cross_df = pd.DataFrame(cross_rows)

print(type_share_df.shape, bucket_df.shape, cross_df.shape)


In [ ]:
drift_summary = drift_summary.sort_values('segment_id').copy()
drift_summary['fraud_rate_prev'] = drift_summary['fraud_rate'].shift(1)
drift_summary['fraud_rate_rel_change'] = (
    (drift_summary['fraud_rate'] - drift_summary['fraud_rate_prev'])
    / drift_summary['fraud_rate_prev'].replace(0, np.nan)
)
drift_summary['label_drift_warning'] = drift_summary['fraud_rate_rel_change'].abs() > 0.20

share_pivot = type_share_df.pivot(index='segment_id', columns='type', values='share').sort_index()
share_diff = share_pivot.diff().abs()
max_share_diff = share_diff.max(axis=1)

drift_summary = drift_summary.merge(max_share_diff.rename('max_type_share_diff').reset_index(), on='segment_id', how='left')
drift_summary['category_drift_warning'] = drift_summary['max_type_share_diff'] > 0.05

for metric in ['amount_log1p_median', 'amount_log1p_p90']:
    prev = drift_summary[metric].shift(1)
    rel = (drift_summary[metric] - prev) / prev.replace(0, np.nan)
    drift_summary[f'{metric}_rel_change'] = rel

drift_summary['amount_drift_warning'] = (
    drift_summary['amount_log1p_median_rel_change'].abs().fillna(0) > 0.15
) | (
    drift_summary['amount_log1p_p90_rel_change'].abs().fillna(0) > 0.15
)

drift_summary[['segment_id','label_drift_warning','category_drift_warning','amount_drift_warning']].head()


In [ ]:
summary_path = OUT_DIR / 'drift_summary_fine_segments.csv'
type_share_path = OUT_DIR / 'drift_type_share.csv'
bucket_path = OUT_DIR / 'drift_amount_bucket.csv'
cross_path = OUT_DIR / 'drift_flag_fraud_cross.csv'

summary_export = drift_summary.merge(cross_df, on='segment_id', how='left')
summary_export.to_csv(summary_path, index=False)
type_share_df.to_csv(type_share_path, index=False)
bucket_df.to_csv(bucket_path, index=False)
cross_df.to_csv(cross_path, index=False)

print(summary_path)


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(drift_summary['segment_id'], drift_summary['fraud_rate'], marker='o')
plt.title('Fraud Rate by Segment')
plt.xlabel('segment_id')
plt.ylabel('fraud_rate')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fraud_rate_by_segment.png', dpi=150)
plt.show()

plt.figure(figsize=(10,4))
plt.plot(drift_summary['segment_id'], drift_summary['fraud_rate'], marker='o', label='fraud_rate')
plt.plot(drift_summary['segment_id'], drift_summary['fraud_rate_roll3'], marker='s', label='rolling_mean_3')
plt.title('Fraud Rate Rolling Mean')
plt.xlabel('segment_id')
plt.ylabel('fraud_rate')
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'fraud_rate_rolling.png', dpi=150)
plt.show()

plt.figure(figsize=(10,5))
for c in ['amount_log1p_median','amount_log1p_p90','amount_log1p_p99']:
    plt.plot(drift_summary['segment_id'], drift_summary[c], marker='o', label=c)
plt.title('log1p(amount) Quantiles by Segment')
plt.xlabel('segment_id')
plt.ylabel('value')
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'amount_quantiles_by_segment.png', dpi=150)
plt.show()

share_plot = type_share_df.pivot(index='segment_id', columns='type', values='share').fillna(0)
share_plot.plot(kind='area', stacked=True, figsize=(10,5), alpha=0.85)
plt.title('Type Share by Segment')
plt.xlabel('segment_id')
plt.ylabel('share')
plt.legend(title='type', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'type_share_by_segment.png', dpi=150)
plt.show()


In [ ]:
holdout_md = f'''# Holdout Definition

- Strategy: last {int(HOLDOUT_RATIO*100)}% unique steps as stable test holdout
- holdout_start_step: {holdout_start_step}
- max_step: {int(unique_steps.max())}

Rows with `step >= {holdout_start_step}` belong to final holdout.
'''
(OUT_DIR / 'holdout_definition.md').write_text(holdout_md, encoding='utf-8')

warn_cols = ['label_drift_warning','category_drift_warning','amount_drift_warning']
warn_table = drift_summary[['segment_id'] + warn_cols].copy()
warn_hits = {c:int(warn_table[c].fillna(False).sum()) for c in warn_cols}
trigger_segments = drift_summary.loc[drift_summary[warn_cols].any(axis=1), 'segment_id'].tolist()

findings_md = f'''# Drift Findings (Auto-generated)

## Warning Counts
- Label drift warnings: {warn_hits['label_drift_warning']}
- Category drift warnings: {warn_hits['category_drift_warning']}
- Amount drift warnings: {warn_hits['amount_drift_warning']}

## Triggered Segments
{trigger_segments}

## Practical Recommendations
1. If warnings cluster in recent segments, recalibrate threshold by recent window.
2. Monitor type-share and amount quantile shifts as online drift signals.
3. Keep final holdout fixed for fair model comparison.
'''
(OUT_DIR / 'drift_findings.md').write_text(findings_md, encoding='utf-8')

print('saved markdown artifacts')


## Notes

- Quantile 採每段抽樣近似，降低記憶體壓力。
- 若要精確 quantile，可改全量計算（成本更高）。